# Scaling Ambiguity Augmenting Human Annotation in Speech Emotion Recognition with Audio-Language Models - Approximated Distributions Preparation

This notebook converts raw **ALM-generated synthetic annotations** into probability
distributions suitable for Ambiguous Emotion Recognition (AER) fine-tuning.

## Pipeline Overview
-----------------
Raw ALM annotations are generated as repeated single-label outputs per
utterance, e.g.:
    {"Speaker XX_Session XX": ["happy", "happy", "sad", "neutral", "sad"]}

This script processes them in three stages:

  Stage 1 — **AnnotatorReducer**
    Truncates the annotation list to a fixed number of annotators (N).
    Useful for the saturation analysis in **Section 4.1** of the paper,
    where we vary N to study the effect on JS divergence.

  Stage 2 — **IEMOAnnotatorRestructurer** / **MSPAnnotatorRestructurer**
    Converts the truncated annotation list into the dataset-specific format
    (IEMOCAP or MSP-Podcast), merging metadata (speaker, audio path,
    groundtruth, need_prediction) from a reference file if provided.
    Keeps emotions in list format at this stage.

  Stage 3 — **RawAnnotationConverter**
    Converts the emotion list into a probability distribution:
        ["happy", "happy", "sad", "sad"] -> {"happy": 0.5, "sad": 0.5}
    This produces the Synthetic Perceptual Proxies used for fine-tuning.

All three stages provide an interactive Jupyter widget UI for exploratory
use; **RawAnnotationConverter** also supports direct programmatic usage.

## Dependencies Import

In [2]:
import json
import ipywidgets as widgets
from IPython.display import display
import os
import re
from collections import Counter
import numpy as np

## Stage 1: AnnotatorReducer
Truncates annotation lists to a fixed N for saturation analysis.

Input format:  ``{sample_id: [{"emotion": "happy", ...}, ...]}``

Output format: same structure, but each list is capped at N entries

In [ ]:
class AnnotatorReducer:
    
    """
    Interactive tool to truncate annotation lists to a fixed number of
    annotators per sample.

    Used in the saturation analysis (Section 4.1): by varying N, we
    measure how JS divergence between synthetic and human distributions
    changes as a function of annotation count.
    """
    
    def __init__(self):
        self.data = None
        self.setup_ui()
        
    def setup_ui(self):
        # File selection widgets
        self.file_selector = widgets.Text(
            value='',
            placeholder='Enter the path to your JSON file',
            description='JSON File:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='50%')
        )
        
        self.load_button = widgets.Button(
            description='Load File',
            button_style='primary',
            tooltip='Click to load the JSON file'
        )
        self.load_button.on_click(self.on_load_button_clicked)
        
        self.output = widgets.Output()
        
        # Annotator count and output widgets
        self.annotator_slider = widgets.IntSlider(
            value=10,
            min=1,
            max=20,
            step=1,
            description='Number of Annotators to Keep:',
            disabled=True,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            style={'description_width': 'initial'}
        )
        
        self.output_file = widgets.Text(
            value='reduced_annotations.json',
            placeholder='Enter output file name',
            description='Output File:',
            disabled=True,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='50%')
        )
        
        self.save_button = widgets.Button(
            description='Save Reduced JSON',
            button_style='success',
            tooltip='Click to save the reduced JSON file',
            disabled=True
        )
        self.save_button.on_click(self.on_save_button_clicked)
        
        self.save_output = widgets.Output()
        
        # Distribution analysis widgets
        self.analyze_button = widgets.Button(
            description='Analyze Distributions',
            button_style='info',
            tooltip='Compare emotion distributions',
            disabled=True
        )
        self.analyze_button.on_click(lambda b: self.analyze_distributions())
        
        self.analysis_output = widgets.Output()
        
        # Display all widgets
        display(widgets.VBox([
            widgets.Label(value="Step 1: Select and load your JSON file"),
            self.file_selector,
            self.load_button,
            self.output,
            widgets.Label(value="Step 2: Select number of annotators to keep and save"),
            self.annotator_slider,
            self.output_file,
            self.save_button,
            self.save_output,
            widgets.Label(value="Step 3: Analyze emotion distributions"),
            self.analyze_button,
            self.analysis_output
        ]))
    
    def load_json_file(self, file_path):
        """Load the JSON file containing annotations."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data
        except Exception as e:
            print(f"Error loading JSON file: {e}")
            return None
    
    def on_load_button_clicked(self, b):
        with self.output:
            self.output.clear_output()
            file_path = self.file_selector.value
            if not file_path:
                print("Please enter a file path")
                return
            
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return
            
            self.data = self.load_json_file(file_path)
            if self.data:
                print(f"Successfully loaded JSON file with {len(self.data)} samples")
                print(f"Each sample has {len(next(iter(self.data.values())))} annotators")
                # Enable the next widgets after successful load
                self.annotator_slider.disabled = False
                self.output_file.disabled = False
                self.save_button.disabled = False
                self.analyze_button.disabled = False
            else:
                print("Failed to load JSON file")
    
    def on_save_button_clicked(self, b):
        with self.save_output:
            self.save_output.clear_output()
            if self.data is None:
                print("No data loaded. Please load a JSON file first.")
                return
            
            num_annotators = self.annotator_slider.value
            output_path = self.output_file.value
            
            # Truncate each sample's annotation list to the selected count
            reduced_data = {}
            for sample_id, annotations in self.data.items():
                reduced_data[sample_id] = annotations[:num_annotators]
            
            try:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(reduced_data, f, indent=2, ensure_ascii=False)
                print(f"Successfully saved reduced annotations to {output_path}")
                print(f"Each sample now has {num_annotators} annotators")
                
                # Display a sample of the reduced data
                sample_id = next(iter(reduced_data.keys()))
                print(f"\nSample preview for '{sample_id}':")
                print(f"Original count: {len(self.data[sample_id])}")
                print(f"Reduced count: {len(reduced_data[sample_id])}")
            except Exception as e:
                print(f"Error saving JSON file: {e}")
    
    def calculate_emotion_distribution(self, annotations):
        """
        Compute the relative frequency of each emotion label in an
        annotation list (used for preview/analysis only).
        """
        emotions = {}
        for ann in annotations:
            emotion = ann['emotion']
            emotions[emotion] = emotions.get(emotion, 0) + 1
        
        # Convert counts to percentages
        total = len(annotations)
        distribution = {emotion: count/total*100 for emotion, count in emotions.items()}
        return distribution
    
    def analyze_distributions(self):
        """
        Print a side-by-side comparison of the full vs. truncated emotion
        distributions for the first few samples.
        """
        with self.analysis_output:
            self.analysis_output.clear_output()
            if self.data is None:
                print("No data loaded. Please load a JSON file first.")
                return

            num_annotators = self.annotator_slider.value

            print(f"Emotion Distribution Analysis (with {num_annotators} annotators)")
            print("-" * 50)

            for i, (sample_id, annotations) in enumerate(self.data.items()):
                reduced_annotations = annotations[:num_annotators]
                original_dist = self.calculate_emotion_distribution(annotations)
                reduced_dist = self.calculate_emotion_distribution(reduced_annotations)

                print(f"\nSample: {sample_id}")
                print("Original distribution:")
                for emotion, percentage in original_dist.items():
                    print(f"  {emotion}: {percentage:.1f}%")

                print("Reduced distribution:")
                for emotion, percentage in reduced_dist.items():
                    print(f"  {emotion}: {percentage:.1f}%")

                # Limit preview to 3 samples to avoid cluttering the output
                if i >= 2:
                    print("\n... (showing only first 3 samples)")
                    break

## Stage 2a: IEMOAnnotatorRestructurer (IEMOCAP)
Converts truncated annotation lists to IEMOCAP dataset format.

Input format:  ``{sample_id: [{"emotion": "happy", ...}, ...]}``

Output format: ``[{"id": ..., "emotion": ["happy", "sad", ...],  
                 "speaker": ..., "audio": ..., "groundtruth": ...,  
                 "need_prediction": ...}]``  
                 
Emotions are still kept as a list here; distribution conversion happens in Stage 3 (RawAnnotationConverter).

In [ ]:
class IEMOAnnotatorRestructurer:
    
    """
    Interactive tool to restructure raw IEMOCAP annotations into the
    per-sample list format expected by the fine-tuning pipeline.

    Optionally merges metadata (speaker, audio path, groundtruth,
    need_prediction) from an existing IEMOCAP-format reference file.
    Also supports combining emotions from both the raw annotation file
    and the reference file (e.g., merging synthetic + human labels).
    """
    
    def __init__(self):
        self.data = None
        self.reference_data = None
        self.setup_ui()
        
    def setup_ui(self):
        # Main file selection widgets
        self.file_selector = widgets.Text(
            value='',
            placeholder='Enter the path to your JSON file with annotator data',
            description='Annotator JSON:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='70%')
        )
        
        self.load_button = widgets.Button(
            description='Load File',
            button_style='primary',
            tooltip='Click to load the JSON file'
        )
        self.load_button.on_click(self.on_restructurer_load_button_clicked)
        
        self.output = widgets.Output()
        
        # Reference file selection widgets
        self.reference_selector = widgets.Text(
            value='',
            placeholder='Enter the path to existing IEMOCAP format JSON file',
            description='Reference IEMOCAP JSON:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='70%')
        )
        
        self.load_reference_button = widgets.Button(
            description='Load Reference',
            button_style='info',
            tooltip='Click to load the reference IEMOCAP JSON file'
        )
        self.load_reference_button.on_click(self.on_load_reference_clicked)
        
        self.reference_output = widgets.Output()
        
        # Restructure widgets
        self.annotator_count = widgets.IntSlider(
            value=3,
            min=1,
            max=20,
            step=1,
            description='Annotators to include:',
            disabled=True,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            style={'description_width': 'initial'}
        )
        
        self.use_reference_metadata = widgets.Checkbox(
            value=True,
            description='Use metadata from reference file (need_prediction, speaker, groundtruth, audio)',
            disabled=True
        )
        
        self.combine_emotions = widgets.Checkbox(
            value=False,
            description='Combine emotions from both files',
            disabled=True
        )
        
        self.need_prediction_dropdown = widgets.Dropdown(
            options=['yes', 'no'],
            value='yes',
            description='need_prediction (if no reference):',
            disabled=True,
            style={'description_width': 'initial'}
        )
        
        self.output_file = widgets.Text(
            value='restructured_annotations.json',
            placeholder='Enter output file name for restructured data',
            description='Output File:',
            disabled=True,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='50%')
        )
        
        self.save_button = widgets.Button(
            description='Restructure & Save',
            button_style='success',
            tooltip='Click to restructure and save to IEMOCAP format',
            disabled=True
        )
        self.save_button.on_click(self.on_save_button_clicked)
        
        self.save_output = widgets.Output()
        
        # Display widgets
        display(widgets.HTML("<h2>Annotator Restructurer: Convert to IEMOCAP format</h2>"))
        display(widgets.VBox([
            widgets.Label(value="Step 1: Select and load your annotator JSON file"),
            self.file_selector,
            self.load_button,
            self.output,
            widgets.Label(value="Step 2: Load reference IEMOCAP format file"),
            self.reference_selector,
            self.load_reference_button,
            self.reference_output,
            widgets.Label(value="Step 3: Configure restructuring settings"),
            self.annotator_count,
            self.use_reference_metadata,
            self.combine_emotions,
            self.need_prediction_dropdown,
            widgets.Label(value="Step 4: Save restructured data"),
            self.output_file,
            self.save_button,
            self.save_output
        ]))
    
    def load_json_file(self, file_path):
        """Load the JSON file containing annotations."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data
        except Exception as e:
            print(f"Error loading JSON file: {e}")
            return None
    
    def on_restructurer_load_button_clicked(self, b):
        with self.output:
            self.output.clear_output()
            file_path = self.file_selector.value
            if not file_path:
                print("Please enter a file path")
                return
            
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return
            
            self.data = self.load_json_file(file_path)
            if self.data:
                print(f"Successfully loaded JSON file with {len(self.data)} samples")
                first_sample = next(iter(self.data.values()))
                print(f"Each sample has {len(first_sample)} annotators")
                
                # Enable controls
                self.annotator_count.disabled = False
                self.need_prediction_dropdown.disabled = False
                self.output_file.disabled = False
                self.save_button.disabled = False
                self.use_reference_metadata.disabled = False
                
                # Update max value for slider based on actual data
                self.annotator_count.max = len(first_sample)
            else:
                print("Failed to load JSON file")
    
    def on_load_reference_clicked(self, b):
        with self.reference_output:
            self.reference_output.clear_output()
            file_path = self.reference_selector.value
            if not file_path:
                print("Please enter a reference file path")
                return
            
            if not os.path.exists(file_path):
                print(f"Reference file not found: {file_path}")
                return
            
            # Load IEMOCAP format reference file
            ref_data = self.load_json_file(file_path)
            
            if ref_data:
                # Convert list format to dictionary with id as key for easier matching
                self.reference_data = {item["id"]: item for item in ref_data if "id" in item}
                
                print(f"Successfully loaded reference file with {len(self.reference_data)} samples")
                
                # Enable the combine emotions checkbox if both files are loaded
                if self.data:
                    self.combine_emotions.disabled = False

                    # Warn if IDs don't align between annotation and reference files
                    main_ids = set(self.data.keys())
                    ref_ids = set(self.reference_data.keys())
                    matching_ids = main_ids.intersection(ref_ids)
                    print(f"Found {len(matching_ids)} matching IDs between files")

                    if len(matching_ids) == 0:
                        print("No matching IDs found between files!")
                    elif len(matching_ids) < len(main_ids):
                        print(f"{len(main_ids) - len(matching_ids)} samples have no matching reference data")
            else:
                print("Failed to load reference file")
    
    def extract_speaker_info(self, sample_id):
        """Extract session and speaker information from sample ID."""
        match = re.match(r'Ses(\d+)([MF])_.*', sample_id)
        if match:
            session_num, gender = match.groups()
            return f"Ses{session_num}_{gender}"
        return "Unknown"
    
    def construct_audio_path(self, sample_id):
        """Construct audio file path based on sample ID."""
        match = re.match(r'(Ses\d+[MF])_(.*?)_([MF]\d+)', sample_id)
        if match:
            prefix, middle, suffix = match.groups()
            session_num = re.search(r'Ses(\d+)', prefix).group(1)
            return f"IEMOCAP_full_release/Session{session_num}/sentences/wav/{prefix}_{middle}/{sample_id}.wav"
        return f"unknown_path/{sample_id}.wav"
    
    def restructure_data(self, num_annotators, use_reference, combine_emotions, need_prediction_value):
        
        """
        Restructure raw annotation dict into IEMOCAP list format.

        For each sample:
          - Extract the top-N emotion labels as a list
          - Optionally merge metadata from the reference file
          - Optionally combine emotions from both sources
          - Fall back to auto-generated metadata if reference is unavailable
        """
        
        if not self.data:
            return None
            
        restructured_data = []
        
        for sample_id, annotations in self.data.items():
            # Get emotions from annotators
            emotions = [ann["emotion"] for ann in annotations[:num_annotators]]
            
            # Create base restructured item
            restructured_item = {
                "id": sample_id,
                "emotion": emotions
            }
            
            # Add metadata from reference file if available and requested
            if use_reference and self.reference_data and sample_id in self.reference_data:
                ref_item = self.reference_data[sample_id]
                
                # Optionally combine emotions from both files
                if combine_emotions and "emotion" in ref_item:
                    # Combine emotions from both sources
                    restructured_item["emotion"] = emotions + ref_item["emotion"]
                
                # Copy metadata fields
                for field in ["need_prediction", "speaker", "groundtruth", "audio"]:
                    if field in ref_item:
                        restructured_item[field] = ref_item[field]
            
            # If no reference data or not using it, generate metadata
            if "need_prediction" not in restructured_item:
                restructured_item["need_prediction"] = need_prediction_value
            
            if "speaker" not in restructured_item:
                restructured_item["speaker"] = self.extract_speaker_info(sample_id)
                
            if "audio" not in restructured_item:
                restructured_item["audio"] = self.construct_audio_path(sample_id)
                
            # Groundtruth is optional, only add if we have it from reference
            if "groundtruth" not in restructured_item and use_reference:
                restructured_item["groundtruth"] = ""
            
            restructured_data.append(restructured_item)
            
        return restructured_data
    
    def on_save_button_clicked(self, b):
        with self.save_output:
            self.save_output.clear_output()
            if self.data is None:
                print("No data loaded. Please load a JSON file first.")
                return
            
            num_annotators = self.annotator_count.value
            use_reference = self.use_reference_metadata.value
            combine_emotions = self.combine_emotions.value
            need_prediction_value = self.need_prediction_dropdown.value
            output_path = self.output_file.value
            
            # Check if reference is needed but not loaded
            if use_reference and not self.reference_data:
                print("You selected to use reference metadata, but no reference file is loaded.")
                print("Please load a reference file or uncheck 'Use metadata from reference file'.")
                return
                
            # Restructure data
            restructured_data = self.restructure_data(
                num_annotators, 
                use_reference, 
                combine_emotions,
                need_prediction_value
            )
            
            if not restructured_data:
                print("Failed to restructure data.")
                return
                
            try:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(restructured_data, f, indent=2, ensure_ascii=False)
                print(f"Successfully saved restructured data to {output_path}")
                print(f"Converted {len(restructured_data)} samples to IEMOCAP format")
                
                # Count emotions in first sample
                first_sample = restructured_data[0]
                num_emotions = len(first_sample.get("emotion", []))
                print(f"Each sample has {num_emotions} emotions in its list")
                
                # Display a sample of the restructured data
                print("\nSample preview of restructured data:")
                print(json.dumps(restructured_data[0], indent=2))
                
                # Provide a summary of the operation
                if use_reference and self.reference_data:
                    matched_count = sum(1 for item in restructured_data if item["id"] in self.reference_data)
                    print(f"\nSummary: {matched_count} of {len(restructured_data)} samples had matching reference data")
                    
                    if combine_emotions:
                        print(f"Emotions were combined from both annotator and reference files")
                    else:
                        print(f"Only emotions from annotator file were used")
            except Exception as e:
                print(f"Error saving restructured data: {e}")

## Stage 2b: MSPAnnotatorRestructurer (MSP-Podcast)
Same purpose as IEMOAnnotatorRestructurer but handles MSP-Podcast ID format.

MSP sample IDs follow: ``MSP-PODCAST_{speaker_id}_{utterance_id}``

Speaker info and audio paths are derived accordingly.

In [ ]:
class MSPAnnotatorRestructurer:
    
    """
    Interactive tool to restructure raw MSP-Podcast annotations into the
    per-sample list format expected by the fine-tuning pipeline.

    Mirrors AnnotatorRestructurer but handles MSP-Podcast-specific
    ID conventions for speaker extraction and audio path construction.
    """
    
    def __init__(self):
        self.data = None
        self.reference_data = None
        self.setup_ui()
        
    def setup_ui(self):
        # Main file selection widgets
        self.file_selector = widgets.Text(
            value='',
            placeholder='Enter the path to your JSON file with LLM annotator data',
            description='LLM Annotations JSON:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='70%')
        )
        
        self.load_button = widgets.Button(
            description='Load File',
            button_style='primary',
            tooltip='Click to load the JSON file'
        )
        self.load_button.on_click(self.on_restructurer_load_button_clicked)
        
        self.output = widgets.Output()
        
        # Reference file selection widgets
        self.reference_selector = widgets.Text(
            value='',
            placeholder='Enter the path to existing MSP format JSON file',
            description='Reference MSP JSON:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='70%')
        )
        
        self.load_reference_button = widgets.Button(
            description='Load Reference',
            button_style='info',
            tooltip='Click to load the reference MSP JSON file'
        )
        self.load_reference_button.on_click(self.on_load_reference_clicked)
        
        self.reference_output = widgets.Output()
        
        # Restructure widgets
        self.annotator_count = widgets.IntSlider(
            value=8,
            min=1,
            max=20,
            step=1,
            description='Annotators to include:',
            disabled=True,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            style={'description_width': 'initial'}
        )
        
        self.use_reference_metadata = widgets.Checkbox(
            value=True,
            description='Use metadata from reference file (need_prediction, speaker, groundtruth, audio)',
            disabled=True
        )
        
        self.combine_emotions = widgets.Checkbox(
            value=False,
            description='Combine emotions from both files',
            disabled=True
        )
        
        self.need_prediction_dropdown = widgets.Dropdown(
            options=['yes', 'no'],
            value='yes',
            description='need_prediction (if no reference):',
            disabled=True,
            style={'description_width': 'initial'}
        )
        
        self.output_file = widgets.Text(
            value='msp_llm_annotations.json',
            placeholder='Enter output file name for restructured data',
            description='Output File:',
            disabled=True,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='50%')
        )
        
        self.save_button = widgets.Button(
            description='Restructure & Save',
            button_style='success',
            tooltip='Click to restructure and save to MSP format',
            disabled=True
        )
        self.save_button.on_click(self.on_save_button_clicked)
        
        self.save_output = widgets.Output()
        
        # Display widgets
        display(widgets.HTML("<h2>MSP Annotator Restructurer: Convert to MSP format</h2>"))
        display(widgets.VBox([
            widgets.Label(value="Step 1: Select and load your LLM annotator JSON file"),
            self.file_selector,
            self.load_button,
            self.output,
            widgets.Label(value="Step 2: Load reference MSP format file (optional)"),
            self.reference_selector,
            self.load_reference_button,
            self.reference_output,
            widgets.Label(value="Step 3: Configure restructuring settings"),
            self.annotator_count,
            self.use_reference_metadata,
            self.combine_emotions,
            self.need_prediction_dropdown,
            widgets.Label(value="Step 4: Save restructured data"),
            self.output_file,
            self.save_button,
            self.save_output
        ]))
    
    def load_json_file(self, file_path):
        """Load the JSON file containing annotations."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data
        except Exception as e:
            print(f"Error loading JSON file: {e}")
            return None
    
    def on_restructurer_load_button_clicked(self, b):
        with self.output:
            self.output.clear_output()
            file_path = self.file_selector.value
            if not file_path:
                print("Please enter a file path")
                return
            
            if not os.path.exists(file_path):
                print(f"File not found: {file_path}")
                return
            
            self.data = self.load_json_file(file_path)
            if self.data:
                print(f"Successfully loaded JSON file with {len(self.data)} samples")
                first_id = next(iter(self.data.keys()))
                first_sample = self.data[first_id]
                print(f"Each sample has {len(first_sample)} annotators")
                
                # Enable controls
                self.annotator_count.disabled = False
                self.need_prediction_dropdown.disabled = False
                self.output_file.disabled = False
                self.save_button.disabled = False
                self.use_reference_metadata.disabled = False
                
                # Update max value for slider based on actual data
                self.annotator_count.max = len(first_sample)
            else:
                print("Failed to load JSON file")
    
    def on_load_reference_clicked(self, b):
        with self.reference_output:
            self.reference_output.clear_output()
            file_path = self.reference_selector.value
            if not file_path:
                print("Please enter a reference file path")
                return
            
            if not os.path.exists(file_path):
                print(f"Reference file not found: {file_path}")
                return
            
            # Load MSP format reference file
            ref_data = self.load_json_file(file_path)
            
            if ref_data:
                # Convert list format to dictionary with id as key for easier matching
                self.reference_data = {}
                for item in ref_data:
                    if "id" in item:
                        self.reference_data[item["id"]] = item
                
                print(f"Successfully loaded reference file with {len(self.reference_data)} samples")
                
                # Enable the combine emotions checkbox if both files are loaded
                if self.data:
                    self.combine_emotions.disabled = False
                
                # Check for ID matching between the two files
                if self.data:
                    main_ids = set(self.data.keys())
                    ref_ids = set(self.reference_data.keys())
                    
                    matching_ids = main_ids.intersection(ref_ids)
                    print(f"Found {len(matching_ids)} matching IDs between files")
                    
                    if len(matching_ids) == 0:
                        print("No matching IDs found between files!")
                    elif len(matching_ids) < len(main_ids):
                        print(f"{len(main_ids) - len(matching_ids)} samples from main file have no matching reference data")
            else:
                print("Failed to load reference file")
    
    def extract_speaker_info(self, sample_id):
        """Extract speaker information from MSP sample ID."""
        match = re.match(r'MSP-PODCAST_(\d+)_\d+', sample_id)
        if match:
            return match.group(1)
        return "Unknown"
    
    def construct_audio_path(self, sample_id):
        """Construct audio file path based on MSP sample ID."""
        return f"Audio/{sample_id}.wav"
    
    def restructure_data(self, num_annotators, use_reference, combine_emotions, need_prediction_value):
        
        """
        Restructure raw annotation dict into MSP-Podcast list format.

        For each sample:
          - Extract the top-N emotion labels as a list
          - Optionally merge metadata from the reference file
          - Optionally combine emotions from both sources
          - Fall back to auto-generated metadata if reference is unavailable
        """
        
        if not self.data:
            return None
            
        restructured_data = []
        
        for sample_id, annotations in self.data.items():
            # Get emotions from annotators (just the emotion field from each annotator)
            emotions = [ann["emotion"] for ann in annotations[:num_annotators]]
            
            # Create base restructured item
            restructured_item = {
                "id": sample_id,
                "emotion": emotions
            }
            
            # Add metadata from reference file if available and requested
            if use_reference and self.reference_data and sample_id in self.reference_data:
                ref_item = self.reference_data[sample_id]
                
                # Optionally combine emotions from both files
                if combine_emotions and "emotion" in ref_item:
                    # Combine emotions from both sources
                    restructured_item["emotion"] = emotions + ref_item["emotion"]
                
                # Copy metadata fields
                for field in ["need_prediction", "speaker", "groundtruth", "audio"]:
                    if field in ref_item:
                        restructured_item[field] = ref_item[field]
            
            # If no reference data or not using it, generate metadata
            if "need_prediction" not in restructured_item:
                restructured_item["need_prediction"] = need_prediction_value
            
            if "speaker" not in restructured_item:
                restructured_item["speaker"] = self.extract_speaker_info(sample_id)
                
            if "audio" not in restructured_item:
                restructured_item["audio"] = self.construct_audio_path(sample_id)
                
            # If groundtruth is missing, add empty array
            if "groundtruth" not in restructured_item:
                restructured_item["groundtruth"] = [""]
            
            restructured_data.append(restructured_item)
            
        return restructured_data
    
    def on_save_button_clicked(self, b):
        with self.save_output:
            self.save_output.clear_output()
            if self.data is None:
                print("No data loaded. Please load a JSON file first.")
                return
            
            num_annotators = self.annotator_count.value
            use_reference = self.use_reference_metadata.value
            combine_emotions = self.combine_emotions.value
            need_prediction_value = self.need_prediction_dropdown.value
            output_path = self.output_file.value
            
            # Check if reference is needed but not loaded
            if use_reference and not self.reference_data:
                print("⚠️ You selected to use reference metadata, but no reference file is loaded.")
                print("Please load a reference file or uncheck 'Use metadata from reference file'.")
                return
                
            # Restructure data
            restructured_data = self.restructure_data(
                num_annotators, 
                use_reference, 
                combine_emotions,
                need_prediction_value
            )
            
            if not restructured_data:
                print("Failed to restructure data.")
                return
                
            # Save the restructured data to a file
            try:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(restructured_data, f, indent=2, ensure_ascii=False)
                print(f"Successfully saved restructured data to {output_path}")
                print(f"Converted {len(restructured_data)} samples to MSP format")
                
                # Count emotions in first sample
                first_sample = restructured_data[0]
                num_emotions = len(first_sample.get("emotion", []))
                print(f"Each sample has {num_emotions} emotions in its list")
                
                # Display a sample of the restructured data
                print("\nSample preview of restructured data:")
                print(json.dumps(restructured_data[0], indent=2))
                
                # Provide a summary of the operation
                if use_reference and self.reference_data:
                    matched_count = sum(1 for item in restructured_data if item["id"] in self.reference_data)
                    print(f"\nSummary: {matched_count} of {len(restructured_data)} samples had matching reference data")
                    
                    if combine_emotions:
                        print(f"Emotions were combined from both annotator and reference files")
                    else:
                        print(f"Only emotions from annotator file were used")
            except Exception as e:
                print(f"Error saving restructured data: {e}")

## Stage 3: RawAnnotationConverter
Converts emotion lists into probability distributions (Synthetic Proxies).

Input format:  ``[{"id": ..., "emotion": ["happy", "happy", "sad"], ...}]``

Output format: ``[{"id": ..., "emotion": {"happy": 0.667, "sad": 0.333},
                  "num_annotators": 3, "annotation_source": ..., ...}]``

This is the final step producing the Synthetic Perceptual Proxies used for fine-tuning in **Section 2.4** of the paper.

In [ ]:
class RawAnnotationConverter:
    
    """
    Converts emotion annotations from list format to probability distribution
    format, producing the Synthetic Perceptual Proxies described in the paper.

    Input:  {"emotion": ["happy", "happy", "sad", "sad"]}
    Output: {"emotion": {"happy": 0.5, "sad": 0.5}, "num_annotators": 4}

    Can be used programmatically (without Jupyter widgets).
    """
    
    def __init__(self, annotation_source="raw-annotations", default_num_annotators=None):
        
        """
        Initialize the converter.
        
        Parameters:
        - annotation_source: Source identifier for the converted annotations
        - default_num_annotators: Default number of annotators if not computable from data
        """
        
        self.annotation_source = annotation_source
        self.default_num_annotators = default_num_annotators
    
    def convert_single_sample(self, sample):
        
        """
        Convert a single sample from raw annotation format to distribution format.
        
        Parameters:
        - sample: Dictionary containing sample data with emotion as list
        
        Returns:
        - Dictionary with emotion as distribution
        """
        
        if not isinstance(sample, dict):
            raise ValueError("Sample must be a dictionary")

        if 'emotion' not in sample:
            raise ValueError("Sample must contain 'emotion' field")

        converted_sample = sample.copy()
        emotion_data = sample['emotion']
        
        if isinstance(emotion_data, list):
            # Count occurrences of each emotion
            if not emotion_data:
                # Empty list - no annotations available
                converted_sample['emotion'] = {}
                converted_sample['num_annotators'] = 0
            else:
                emotion_counts = Counter(emotion_data)
                total_annotations = len(emotion_data)
                
                # Convert to distribution (probabilities)
                emotion_distribution = {
                    emotion: count / total_annotations 
                    for emotion, count in emotion_counts.items()
                }
                
                converted_sample['emotion'] = emotion_distribution
                converted_sample['num_annotators'] = total_annotations
        
        elif isinstance(emotion_data, dict):
            # Already in distribution format - just add metadata
            converted_sample['num_annotators'] = self.default_num_annotators or len(emotion_data)
        
        else:
            # Single emotion string - convert to distribution
            converted_sample['emotion'] = {emotion_data: 1.0}
            converted_sample['num_annotators'] = 1
        
        # Add annotation source metadata
        converted_sample['annotation_source'] = self.annotation_source
        return converted_sample
    
    def convert_dataset(self, dataset):
        """
        Convert an entire dataset from raw annotation format to distribution format.
        
        Parameters:
        - dataset: List of samples with raw emotion annotations
        
        Returns:
        - List of samples with emotion distributions
        """
        if not isinstance(dataset, list):
            raise ValueError("Dataset must be a list of samples")
        
        converted_dataset = []
        
        for i, sample in enumerate(dataset):
            try:
                converted_sample = self.convert_single_sample(sample)
                converted_dataset.append(converted_sample)
            except Exception as e:
                print(f"Error converting sample {i}: {e}")
                # Skip problematic samples but continue processing
                continue
        
        return converted_dataset
    
    def save_converted_dataset(self, dataset, output_path):
        """
        Convert dataset and save to file.
        
        Parameters:
        - dataset: List of samples with raw emotion annotations
        - output_path: Path to save the converted dataset
        """
        converted_dataset = self.convert_dataset(dataset)
        
        # Create output directory if it doesn't exist
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Save to file
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(converted_dataset, f, indent=2, ensure_ascii=False)
        
        print(f"Converted dataset saved to: {output_path}")
        print(f"Total samples converted: {len(converted_dataset)}")
        
        return converted_dataset
    
    def get_emotion_statistics(self, dataset):
        
        """
        Compute summary statistics for a converted (distribution-format) dataset.

        Returns a dict with:
          - total_samples
          - unique_emotions
          - avg/min/max annotators per sample
          - avg Shannon entropy of emotion distributions
          - annotation_source label

        Parameters
        ----------
        dataset : list of dict
            Output of convert_dataset().
        """
        
        if not dataset:
            return {"error": "Empty dataset"}

        all_emotions = set()
        annotation_counts = []
        entropies = []

        for sample in dataset:
            if 'emotion' in sample and isinstance(sample['emotion'], dict):
                all_emotions.update(sample['emotion'].keys())

                if 'num_annotators' in sample:
                    annotation_counts.append(sample['num_annotators'])

                # Shannon entropy as a measure of distributional ambiguity
                probs = list(sample['emotion'].values())
                if probs:
                    entropy = -sum(p * np.log2(p) for p in probs if p > 0)
                    entropies.append(entropy)

        return {
            "total_samples": len(dataset),
            "unique_emotions": list(all_emotions),
            "num_unique_emotions": len(all_emotions),
            "avg_annotators_per_sample": np.mean(annotation_counts) if annotation_counts else 0,
            "min_annotators": min(annotation_counts) if annotation_counts else 0,
            "max_annotators": max(annotation_counts) if annotation_counts else 0,
            "avg_distribution_entropy": np.mean(entropies) if entropies else 0,
            "annotation_source": self.annotation_source
        }

## Interactive Execution
Instantiate each stage's widget UI.

Run these cells sequentially when working in a notebook environment.

In [ ]:
# Stage 1: Reduce annotation count for saturation analysis
reducer = AnnotatorReducer()

In [ ]:
# Stage 2a: Restructure to IEMOCAP format
restructurer = IEMOAnnotatorRestructurer()

HTML(value='<h2>Annotator Restructurer: Convert to IEMOCAP format</h2>')

In [ ]:
# Stage 2b: Restructure to MSP-Podcast format
msprestructure = MSPAnnotatorRestructurer()

HTML(value='<h2>MSP Annotator Restructurer: Convert to MSP format</h2>')

In [ ]:
# Stage 3: Convert list annotations to probability distributions
# Example: process human/synthetic/combined annotations for MSP-Podcast training set
converter = RawAnnotationConverter(
    annotation_source="combined-human-annotations"
)

with open("combined_msp_train_raw_annotations.json", 'r') as f:
    raw_data = json.load(f)

converted_data = converter.save_converted_dataset(
    raw_data,
    "processed_data/converted_distributions/combined_msp_train_distributions.json"
)

stats = converter.get_emotion_statistics(converted_data)
print("Stats:", stats)


Converted dataset saved to: processed_data/converted_distributions/combined_msp_train_distributions.json
Total samples converted: 3291
Stats: {'total_samples': 3291, '\nunique_emotions': ['Neutral', 'Happy', 'Sad', 'Angry'], '\nnum_unique_emotions': 4, '\navg_annotators_per_sample': np.float64(15.497113339410513), '\nmin_annotators': 15, '\nmax_annotators': 31, '\navg_distribution_entropy': np.float64(0.5515320127289598), '\nannotation_source': 'combined-human-annotations'}
